In [7]:
import torch
import random
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [8]:
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))

s_i = {s:i+1 for i, s in enumerate(chars)}
s_i["."] = 0

i_s = {i:s for s, i in s_i.items()}

# building dataset

block_size = 3

def build_dataset(words):
    
    X, Y = [], []
    for w in words:
        # print(w)
        context = [0] * block_size
        
        for ch in w + ".":
            ix = s_i[ch]
            X.append(context)
            Y.append(ix)
            # print("".join(i_s[i] for i in context), "---->", i_s[ix])
            context = context[1:] + [ix]
            
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [9]:
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [10]:
n_embd = 10 # dimensionality of character embedding vector
n_chars = 27
n_hidden = 200 # number of neurons in hidden layer

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((n_chars, n_embd), generator=g)

w1 = torch.randn((n_embd * block_size, n_hidden)) * (5/3) / (30 ** 0.5) # kaiming initialization
# b1 = torch.randn(n_hidden) * 0.01

w2 = torch.randn((n_hidden, n_chars)) * 0.01
b2 = torch.randn(n_chars) * 0.1

bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))

bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, w1, w2, b2, bngain, bnbias]

for p in parameters:
    p.requires_grad = True

In [11]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10 ** lre

lri = []

def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    
    print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}")

In [12]:
# training loop

max_steps = 200000
batch_size = 32

lossi = []
stepi = []

ix = torch.randint(0, Xtr.shape[0], (batch_size,))
Xb, Yb = Xtr[ix], Ytr[ix]

n = batch_size

# forward pass

# linear layer
# --------------------------------------
emb = C[Xb]
embcat = emb.view(-1, 30)
hpreact = embcat @ w1 # + b1
# --------------------------------------

# batch normalization layer
# --------------------------------------
bnmeani = hpreact.mean(0, keepdim=True)
bnstdi = hpreact.std(0, keepdim=True)

hpreact = bngain * (hpreact - bnmeani) / (bnstdi + 0.001) + bnbias # added an epsilon to bnstdi

with torch.no_grad():
    bnmean_running = 0.99 * bnmean_running + 0.001 * bnmeani

with torch.no_grad():
    bnstd_running = 0.99 * bnstd_running + 0.001 * bnstdi
# --------------------------------------

# non-linear layer
# --------------------------------------
h = torch.tanh(hpreact)

logits = h @ w2 + b2
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()
# --------------------------------------

# backward pass
for p in parameters:
    p.grad = None

loss.backward()

# update
# lr = lrs[i]
lr = 0.01
for p in parameters:
    p.data += -lr * p.grad
    
# track stats  
# lri.append(lre[i])
#if i % 1000 == 0:
#    print(f"{i:7d}/{max_steps:7d}: {loss.item():4f}")

#stepi.append(i)
#lossi.append(loss.log10().item())

In [ ]:
dlogprobs = torch.zeros_like(logprobs)
dprobs = (1.0 / probs) * dlogprobs
dcounts_sum_inv = counts * dprobs
dcounts_sum = -(counts_sum ** -2) * dcounts_sum_inv

dcounts = counts_sum_inv * dprobs
dnorm_logits = norm_logits * dcounts
dlogix